# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [1]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce

Initialize PySpark Context

In [2]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Define function for loading data

In [8]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [10]:
#export
def clean_data(df):
    '''
    input: df a dataframe
    output: df a dataframe with the all the original columns
    '''

    # START YOUR CODE HERE ---------
    columns_to_cast = {
        "passenger_count": "int",
        "total_amount": "float",
        "tip_amount": "float",
        "trip_distance": "float",
        "fare_amount": "float",
        "tpep_pickup_datetime": "timestamp",
        "tpep_dropoff_datetime": "timestamp",
    }

    for col_name, data_type in columns_to_cast.items():
        df = df.withColumn(col_name, df[col_name].cast(data_type))
    # END YOUR CODE HERE -----------

    return df

### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations.

In [31]:
#export
def common_pair(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - PULocationID
            - DOLocationID
            - total_passenger_count
            - per_person_rate

    per_person_rate is the total_amount per person for a given pair.

    '''

    # START YOUR CODE HERE ---------
    from pyspark.sql.functions import sum, desc

    filtered_df = df.filter(col("PULocationID") != col("DOLocationID"))

    aggregated_df = filtered_df.groupBy("PULocationID", "DOLocationID").agg(
        sum("passenger_count").alias("total_passenger_count"),
        sum("total_amount").alias("total_amount_of_trips")
    )

    result_df = aggregated_df.withColumn(
        "per_person_rate",
        col("total_amount_of_trips") / col("total_passenger_count")
    )

    sorted_df = result_df.orderBy(
        desc("total_passenger_count"),
        desc("per_person_rate")
    )

    final_result = sorted_df.select(
        col("PULocationID"),
        col("DOLocationID"),
        col("total_passenger_count"),
        col("per_person_rate")
    ).limit(10)
    # END YOUR CODE HERE -----------

    return final_result

### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [35]:
#export
def distance_with_most_tip(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - trip_distance
            - tip_percent

    trip_percent is the percent of tip out of fare_amount

    '''

    # START YOUR CODE HERE ---------
    from pyspark.sql.functions import col, round, ceil, avg, desc

    filtered_df = df.filter(
        (col("fare_amount") > 2.00) & (col("trip_distance") > 0)
    )

    df_with_tip = filtered_df.withColumn(
        "tip_percent",
        (col("tip_amount") * 100) / col("fare_amount")
    )

    df_with_rounded_dist = df_with_tip.withColumn(
        "trip_distance_rounded",
        ceil(col("trip_distance"))
    ).drop("trip_distance")\
    .withColumnRenamed("trip_distance_rounded", "trip_distance")

    aggregated_df = df_with_rounded_dist.groupBy("trip_distance").agg(
        avg("tip_percent").alias("tip_percent")
    )

    sorted_df = aggregated_df.orderBy(
        desc("tip_percent")
    )

    final_result = sorted_df.select(
        col("trip_distance"),
        col("tip_percent")
    ).limit(15)
    # END YOUR CODE HERE -----------

    return final_result

### Q1.4

Determine the average speed at different times of day.

In [41]:
#export
def time_with_most_traffic(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - time_of_day
            - am_avg_speed
            - pm_avg_speed

    am_avg_speed and pm_avg_speed are the average trip distance / average trip time calculated for each hour

    '''

    # START YOUR CODE HERE ---------
    from pyspark.sql.functions import (
        hour,
        unix_timestamp,
        avg,
        when,
        desc
    )

    df_with_duration = df.withColumn(
        # Calculate duration in seconds using unix_timestamp difference
        "duration_seconds",
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime")))
    ).withColumn(
        # Extract the 24-hour time (0-23)
        "hour_24",
        hour(col("tpep_pickup_datetime"))
    ).withColumn(
        # Determine the AM/PM period
        "time_period",
        when(col("hour_24") < 12, "AM").otherwise("PM")
    ).withColumn(
        # Convert to 12-hour format (0-11) for grouping
        "time_of_day_12",
        when(col("hour_24") < 12, col("hour_24")).otherwise(col("hour_24") - 12)
    )

    # 3. Aggregate Metrics (Average Distance and Average Duration) by 24-hour time

    aggregated_df = df_with_duration.groupBy("hour_24", "time_period", "time_of_day_12").agg(
        avg("trip_distance").alias("avg_distance"),
        avg("duration_seconds").alias("avg_duration_seconds")
    )

    # 4. Calculate Average Speed (Distance per Hour)

    # Formula: Speed = Avg_Distance / (Avg_Duration_Seconds / 3600)
    # Handle potential division by zero for duration_seconds
    calculated_speed_df = aggregated_df.withColumn(
        "avg_speed",
        when(
            col("avg_duration_seconds") > 0,
            # Cast to DoubleType explicitly before division for accurate float result
            (col("avg_distance").cast("double") / (col("avg_duration_seconds") / 3600))
        ).otherwise(0.0) # Speed is 0 if duration is 0, indicating extremely high traffic/short trip
    )

    # 5. Conditional Aggregation (Pivot) to Separate AM and PM Speeds

    # Group by the 12-hour format (0-11)
    final_traffic_df = calculated_speed_df.groupBy("time_of_day_12").agg(
        # Conditional aggregation for AM speed
        avg(when(col("time_period") == "AM", col("avg_speed"))).alias("am_avg_speed"),
        # Conditional aggregation for PM speed
        avg(when(col("time_period") == "PM", col("avg_speed"))).alias("pm_avg_speed")
    )

    # 6. Final Formatting and Sorting

    # Format the final columns: Round speed to 2 decimal places and rename 'time_of_day_12'
    final_result = final_traffic_df.select(
        col("time_of_day_12").alias("time_of_day"),
        col("am_avg_speed").alias("am_avg_speed"),
        col("pm_avg_speed").alias("pm_avg_speed")
    ).orderBy(
        "time_of_day" # Sort ascending by 12-hour time (0-11)
    )
    # END YOUR CODE HERE -----------

    return final_result

## The below cells are for you to investigate your solutions and will not be graded

In [15]:
df = load_data()
df = clean_data(df)

In [32]:
common_pair(df).show()

+------------+------------+---------------------+-----------------+
|PULocationID|DOLocationID|total_passenger_count|  per_person_rate|
+------------+------------+---------------------+-----------------+
|         239|         238|                   62|4.262741935483872|
|         237|         236|                   60|4.482500000000002|
|         263|         141|                   52|3.419038461538462|
|         161|         236|                   42|5.368571428571429|
|         148|          79|                   42|4.711904761904762|
|         142|         238|                   39|5.054871794871795|
|         141|         236|                   37|4.355675675675675|
|         239|         143|                   37|4.252162162162162|
|         239|         142|                   35|3.817714285714285|
|          79|         170|                   34|6.394705882352943|
+------------+------------+---------------------+-----------------+



In [36]:
distance_with_most_tip(df).show()

+-------------+------------------+
|trip_distance|       tip_percent|
+-------------+------------------+
|            1| 17.12981600406835|
|            2|15.815527131655559|
|           17|15.796441782308916|
|           20| 15.11240992123345|
|            3|14.886705733807549|
|            6|14.579695047973713|
|            5|14.245405873948084|
|            4|13.831569492912116|
|            9|13.814476585827178|
|            8| 12.07259674604865|
|           19|11.952632334985276|
|           10| 11.88049048422389|
|            7|10.800575589589778|
|           21|10.739019886973427|
|           18| 10.69682224321948|
+-------------+------------------+



In [42]:
time_with_most_traffic(df).show()

+-----------+------------------+------------------+
|time_of_day|      am_avg_speed|      pm_avg_speed|
+-----------+------------------+------------------+
|          0| 9.377696197864013|              NULL|
|          1|10.845483498407587| 5.125214408233276|
|          3|              NULL|               0.0|
|          4|              NULL|               0.0|
|          5|              NULL|0.5137660219476305|
|          6|              NULL|  9.98984771573604|
|          7|              NULL|0.1841530573490315|
|          8|              NULL|0.5183127584049216|
|         10|              NULL|0.6147483852980808|
|         11|              NULL| 4.650958301847908|
+-----------+------------------+------------------+

